# Frequency-Dependent RIS Optimization Test

We set up a **40×40 RIS at 28 GHz** and compare:
1. **Case A** – `optimize()` without `fc` → uses `ris.fc = 28 GHz`
2. **Case B** – `optimize(fc=26e9)` → explicit 26 GHz

Built-in plot methods used:
- `channel.plot_path_gain_2d_map(...)` — path-gain heatmap
- `ris.plot_phase_dist()` — phase-shift distribution

In [ ]:
import numpy as np
from scipy.constants import c, pi

from surpRISe.channels.tx2ris2rx.tang_general_channel import tang_general_channel
from surpRISe.components.antenna import Antenna
from surpRISe.components.ris import RIS

print('Imports OK')

## 1 – System Setup

In [ ]:
# ── Carrier frequency & wavelength (RIS is designed for 28 GHz) ───────────────
fc_ris = 28e9          # Hz  – RIS centre frequency
lam    = c / fc_ris    # m

# ── RIS: 40×40 elements, boresight along X, half-wavelength spacing ───────────
ris = RIS(
    center            = [0, 0, 0],
    boresight         = 'x',        # RIS faces along +X
    Ncol              = 40,
    Nrow              = 40,
    delta_h           = lam / 2,
    delta_v           = lam / 2,
    dx                = 0.47 * lam,
    dy                = 0.47 * lam,
    fc                = fc_ris,
    radiation_factor  = 1,
    phase_shifts_dict = {'0': 0, '1': pi},  # 1-bit phase shifters
    att_dict          = {'0': 1, '1': 1},
)

# ── TX at [5, 5, 0] and RX at [10, -2, 0] (both in the z=0 plane) ────────────
tx_antenna = Antenna(
    position = [5, 5, 0],
    gain_dbi = 5,
)
rx_antenna = Antenna(
    position = [10, -2, 0],
    gain_dbi = 5,
)

# ── Channel model ─────────────────────────────────────────────────────────────
channel = tang_general_channel()

print(f'RIS: {ris.Nrow}x{ris.Ncol} elements  |  fc_ris = {fc_ris/1e9:.0f} GHz  |  boresight = {ris.boresight}')
print(f'TX position: {tx_antenna.position}')
print(f'RX position: {rx_antenna.position}')

## 2 – Sweep parameters (shared by both cases)

With `boresight='x'` the RIS lies in the YZ-plane.  
Both TX and RX have `z=0`, so we sweep the **XY-plane** (fixed z=0).

In [ ]:
# plane = 'xy'  →  axis1 = X, axis2 = Y, fixed axis = Z = 0
xlim    = (-2, 15)   # m  – X range
ylim    = (-5, 8)    # m  – Y range  (covers TX y=5 and RX y=-2)
Nx, Ny  = 80, 80

# colour-scale limits for the path-gain map (dB)
vmin_db = -160
vmax_db = -80

print(f'Sweep: {Nx}x{Ny} points in xy-plane (z=0)')

## 3 – Case A: optimize at default fc (= ris.fc = 28 GHz)

In [ ]:
ris.reset_ris()

channel.optimize(
    optimizer  = 'direct_disc',
    mode       = 'disc',
    wave_model = 'spherical',
    tx_antenna = tx_antenna,
    rx_antenna = rx_antenna,
    ris        = ris,
    # fc not passed → defaults to ris.fc = 28 GHz
)
print('Case A done (fc = 28 GHz, default)')

In [ ]:
print('Case A – Path-gain map:')
channel.plot_path_gain_2d_map(
    tx_antenna = tx_antenna,
    rx_antenna = rx_antenna,
    ris        = ris,
    ax1lim     = xlim,
    ax2lim     = ylim,
    Nax1       = Nx,
    Nax2       = Ny,
    plane      = 'xy',
    vmin       = vmin_db,
    vmax       = vmax_db,
)

In [ ]:
print('Case A – Phase distribution:')
ris.plot_phase_dist()

## 4 – Case B: optimize at explicit fc = 26 GHz

In [ ]:
fc_opt = 26e9   # Hz

ris.reset_ris()

channel.optimize(
    optimizer  = 'direct_disc',
    mode       = 'disc',
    wave_model = 'spherical',
    tx_antenna = tx_antenna,
    rx_antenna = rx_antenna,
    ris        = ris,
    fc         = fc_opt,    # ← explicit 26 GHz
)
print(f'Case B done (fc = {fc_opt/1e9:.0f} GHz, explicit)')

In [ ]:
print('Case B – Path-gain map:')
channel.plot_path_gain_2d_map(
    tx_antenna = tx_antenna,
    rx_antenna = rx_antenna,
    ris        = ris,
    ax1lim     = xlim,
    ax2lim     = ylim,
    Nax1       = Nx,
    Nax2       = Ny,
    plane      = 'xy',
    vmin       = vmin_db,
    vmax       = vmax_db,
)

In [ ]:
print('Case B – Phase distribution:')
ris.plot_phase_dist()

## 5 – Numeric comparison

In [ ]:
# Re-capture phase matrices for comparison
ris.reset_ris()
channel.optimize(
    optimizer='direct_disc', mode='disc', wave_model='spherical',
    tx_antenna=tx_antenna, rx_antenna=rx_antenna, ris=ris,
)
phase_28 = ris.elem_phase_shift.copy()

ris.reset_ris()
channel.optimize(
    optimizer='direct_disc', mode='disc', wave_model='spherical',
    tx_antenna=tx_antenna, rx_antenna=rx_antenna, ris=ris, fc=fc_opt,
)
phase_26 = ris.elem_phase_shift.copy()

diff_pct = (phase_28 != phase_26).mean() * 100
print(f'Elements with different phase (28 GHz vs 26 GHz optimisation): {diff_pct:.1f}%')